## Populating decoding-related hex maze tables

So you've done decoding for your hex maze session and have a results xarray.
It's time to assign the decoded location to a hex so we can use it for analysis!

In [1]:
import datajoint as dj
import numpy as np

import spyglass.common as sgc
from spyglass.common import Nwbfile
from spyglass.utils.nwb_helper_fn import get_nwb_file

from spyglass_hexmaze.hex_maze_decoding import (
    DecodedPosition,
    DecodedHexPositionSelection,
    DecodedHexPosition,
    DecodedHexPath,
)


/home/scrater/miniforge3/envs/spyglass/lib/python3.10/site-packages/datajoint/plugin.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[2026-02-24 17:07:38,115][INFO]: DataJoint 0.14.6 connected to scrater@lmf-db.cin.ucsf.edu:3306


View existing entries in hex maze decode related tables

In [2]:
display(DecodedPosition())

display(DecodedHexPositionSelection())

display(DecodedHexPosition())

display(DecodedHexPath())


decoding_merge_id,nwb_file_name name of the NWB file,analysis_file_name name of the file,decoded_position_object_id
04cda9c4-b117-800d-9296-2d647212b3c8,Toby20250318_.nwb,Toby20250318_EN9J6XWJ1K.nwb,066c6b3a-0dba-4917-9c40-2359eaa54fd2
198ac087-bdf3-5ef0-e790-12db43cc8641,Toby20250327_.nwb,Toby20250327_58D3V47S5Z.nwb,3ee59985-e644-4cd0-bd5d-3a79a156addf
231ed383-2f43-fc37-7e28-2e1ebce17873,BraveLu20240505_.nwb,BraveLu20240505_BZ1G49CS6S.nwb,a2619f97-ec05-4e51-a833-2eeed0cdbd3d
4c36eef6-a8b0-869c-5d22-a21838d19278,Toby20250316_.nwb,Toby20250316_RI41ZP7EHL.nwb,1c003f3c-c57b-4a96-8014-9d31c920a702
605e3a2e-e4cb-242a-7aa8-01280e833f67,Toby20250318_.nwb,Toby20250318_GXKVQWJZBT.nwb,65e82bb6-d629-481c-b04b-71cf7be5fdf6
72f6bff1-1ec1-4826-628b-d850d6867709,Toby20250318_.nwb,Toby20250318_7DDCYRMN6J.nwb,bb6b0812-19d7-439d-8ecd-462e2421d8a6
804d4fd0-101b-01eb-c124-e1c80c589644,Toby20250316_.nwb,Toby20250316_H7Z7FRQKQU.nwb,213f6f9b-2d52-4e05-9335-574fbb48afa4
aee87e27-ca98-4645-9051-85d52378942c,Toby20250327_.nwb,Toby20250327_YJ23OO4PBR.nwb,c007dcf6-dad4-4a52-8a9f-47ae5d18401a
b94510ec-58c2-a1a9-f212-58b740944172,Toby20250327_.nwb,Toby20250327_CETTIJOITE.nwb,905aef1b-fe01-4efd-8d8d-6b4e2043a0b0
c6932302-5f74-dc1c-4e86-b1c3604817c3,IM-1478_20220725_.nwb,IM-1478_20220725_7N3Y992B6P.nwb,2c451d70-ac95-424c-929c-07f8de159a0c


decoding_merge_id,nwb_file_name name of the NWB file,epoch the session epoch for this task and apparatus(1 based)
04cda9c4-b117-800d-9296-2d647212b3c8,Toby20250318_.nwb,5
198ac087-bdf3-5ef0-e790-12db43cc8641,Toby20250327_.nwb,3
4c36eef6-a8b0-869c-5d22-a21838d19278,Toby20250316_.nwb,5
605e3a2e-e4cb-242a-7aa8-01280e833f67,Toby20250318_.nwb,3
72f6bff1-1ec1-4826-628b-d850d6867709,Toby20250318_.nwb,1
804d4fd0-101b-01eb-c124-e1c80c589644,Toby20250316_.nwb,1
aee87e27-ca98-4645-9051-85d52378942c,Toby20250327_.nwb,1
b94510ec-58c2-a1a9-f212-58b740944172,Toby20250327_.nwb,7
e7b6f591-0016-95ca-3c5c-76426ab81366,Toby20250327_.nwb,5
f28ed7b7-2d6f-6dd5-f0c3-6882c6635eff,Toby20250318_.nwb,7


decoding_merge_id,nwb_file_name name of the NWB file,epoch the session epoch for this task and apparatus(1 based),analysis_file_name name of the file,hex_assignment_object_id
04cda9c4-b117-800d-9296-2d647212b3c8,Toby20250318_.nwb,5,Toby20250318_4MMA1RXC9W.nwb,cfb287b6-c554-4841-b0e6-94803293eb62
198ac087-bdf3-5ef0-e790-12db43cc8641,Toby20250327_.nwb,3,Toby20250327_SYSU5H8DBA.nwb,6ebfae1d-7fd8-43cb-a056-a386afcb0d30
4c36eef6-a8b0-869c-5d22-a21838d19278,Toby20250316_.nwb,5,Toby20250316_KD0M6PI3R7.nwb,a6848677-3ad8-4d3c-b821-74c3ffa7bc08
605e3a2e-e4cb-242a-7aa8-01280e833f67,Toby20250318_.nwb,3,Toby20250318_VQT45L8J0J.nwb,875d0619-59af-4efa-a249-3a63f4f52fd4
72f6bff1-1ec1-4826-628b-d850d6867709,Toby20250318_.nwb,1,Toby20250318_XLIYXF84RM.nwb,a26d340b-aaae-4f0d-ae1f-74af4b3aba5a
804d4fd0-101b-01eb-c124-e1c80c589644,Toby20250316_.nwb,1,Toby20250316_NLC74TYMCA.nwb,01cc038f-927a-41ae-b1a5-ac821910d35e
aee87e27-ca98-4645-9051-85d52378942c,Toby20250327_.nwb,1,Toby20250327_OXJLVNTE1V.nwb,5533c74e-324c-4755-871b-3b23843e33f3
b94510ec-58c2-a1a9-f212-58b740944172,Toby20250327_.nwb,7,Toby20250327_4D8IV2E89M.nwb,ad2b9ce0-4f87-4415-8460-465a7afe33c8
e7b6f591-0016-95ca-3c5c-76426ab81366,Toby20250327_.nwb,5,Toby20250327_T8TCL9KW5L.nwb,072fc04a-1e91-4d46-baeb-8df1fb0bea0d
f28ed7b7-2d6f-6dd5-f0c3-6882c6635eff,Toby20250318_.nwb,7,Toby20250318_MM5YCT6Y5B.nwb,07969451-ddaa-449c-84c2-e65563c58851


decoding_merge_id,nwb_file_name name of the NWB file,epoch the session epoch for this task and apparatus(1 based),analysis_file_name name of the file,hex_path_object_id
f34661fe-39f0-9102-fc76-f337f1f98d1d,Toby20250316_.nwb,3,Toby20250316_ADF7JDGPAU.nwb,488c1cf5-c408-4006-b3d3-3b98aa0f1caa


Grab a `merge_id` that points to the `DecodingOutput` entry you want to use

In [3]:
from spyglass.decoding.decoding_merge import DecodingOutput

# Fetch results to make sure they exist
merge_id = {"merge_id": "f34661fe-39f0-9102-fc76-f337f1f98d1d"}
results = DecodingOutput.fetch_results(merge_id)

display(results)

print(f"Merge id: {merge_id}")

[2026-02-24 17:08:46,122][WARNING]: Skipped checksum for file with hash: 15470b6a-a622-4930-7be1-3ff20b395bc5, and path: /stelmo/nwb/analysis/Toby20250316/Toby20250316_34e77c77-e356-47fd-9938-e92bfa188f54.nc
/home/scrater/miniforge3/envs/spyglass/lib/python3.10/site-packages/xarray/namedarray/core.py:496: UserWarning: Duplicate dimension names present: dimensions {'states'} appear more than once in dims=('states', 'states'). We do not yet support duplicate dimension names, but we do allow initial construction of the object. We recommend you rename the dims immediately to become distinct, as most xarray functionality is likely to fail silently if you do not. To rename the dimensions you will need to set the ``.dims`` attribute of each variable, ``e.g. var.dims=('x0', 'x1')``.
  warnings.warn(
/home/scrater/miniforge3/envs/spyglass/lib/python3.10/site-packages/xarray/namedarray/core.py:496: UserWarning: Duplicate dimension names present: dimensions {'states'} appear more than once in dim

<xarray.Dataset> Size: 60GB
Dimensions:                      (time: 1360969, state_ind: 11088,
                                  dim_0: 11088, states: 2, intervals: 1,
                                  state_bins: 11088)
Coordinates:
  * time                         (time) float64 11MB 1.742e+09 ... 1.742e+09
  * state_ind                    (state_ind) int32 44kB 0 0 0 0 0 ... 1 1 1 1 1
  * states                       (states) object 16B 'Continuous' 'Fragmented'
    environments                 (states) object 16B ...
    encoding_groups              (states) int32 8B ...
  * state_bins                   (state_bins) object 89kB MultiIndex
  * state                        (state_bins) object 89kB 'Continuous' ... 'F...
  * x_position                   (state_bins) float64 89kB 31.25 31.25 ... 182.4
  * y_position                   (state_bins) float64 89kB 7.265 9.257 ... 148.7
Dimensions without coordinates: dim_0, intervals
Data variables:
    initial_conditions           (dim_0) float64 89kB ...
    discrete_state_transitions   (states, states) float64 32B ...
    acausal_posterior            (intervals, time, state_bins) float32 60GB ...
    acausal_state_probabilities  (intervals, time, states) float64 22MB ...
Attributes:
    marginal_log_likelihoods:  -6310602.5

Merge id: {'merge_id': 'f34661fe-39f0-9102-fc76-f337f1f98d1d'}


In [4]:
from non_local_detector.analysis import get_ahead_behind_distance
from spyglass.decoding import SortedSpikesDecodingV1

display(SortedSpikesDecodingV1() & "nwb_file_name LIKE '%IM-%'")

nwb_file_name name of the NWB file,unit_filter_params_name,sorted_spikes_group_name,position_group_name,decoding_param_name a name for this set of parameters,encoding_interval descriptive name of this interval list,decoding_interval descriptive name of this interval list,estimate_decoding_params whether to estimate the decoding parameters,results_path path to the results file,classifier_path path to the classifier file
IM-1478_20220724_.nwb,default_exclusion,sorted_spikes_group,sorted_spikes_pos_group,contfrag_sorted_50chunks_100blocks,00_r1,00_r1,1,=BLOB=,=BLOB=
IM-1478_20220724_.nwb,default_exclusion,sorted_spikes_group,sorted_spikes_pos_group,contfrag_sorted_50chunks_100blocks,00_r1,epoch0_block1,1,=BLOB=,=BLOB=
IM-1478_20220725_.nwb,default_exclusion,sorted_spikes_group,sorted_spikes_pos_group,contfrag_sorted_50chunks_100blocks,00_r1,00_r1,1,=BLOB=,=BLOB=
IM-1478_20220725_.nwb,default_exclusion,sorted_spikes_group,sorted_spikes_pos_group,contfrag_sorted_50chunks_100blocks_2halfbin,00_r1,epoch0_block1,1,=BLOB=,=BLOB=
IM-1594_20230725_.nwb,default_exclusion,sorted_spikes_group,1000uV_artifact_65uV_detection_custom_channels,contfrag_sorted_50chunks_100blocks,00_r1,epoch0_block1,1,=BLOB=,=BLOB=
IM-1941_20260207_.nwb,default_exclusion,sorted_spikes_group,sorted_spikes_pos_group,contfrag_sorted_50chunks_100blocks,00_r1,epoch0_block1,1,=BLOB=,=BLOB=


In [5]:
abd = (SortedSpikesDecodingV1 & {"nwb_file_name": "IM-1478_20220725_.nwb", "decoding_interval": "epoch0_block1"}).get_ahead_behind_distance()

2026-02-24 17:09:13.907371: W external/org_tensorflow/tensorflow/compiler/xla/pjrt/gpu/gpu_helpers.cc:63] Unable to enable peer access between GPUs 0 and 9; status: INTERNAL: failed to enable peer access from 0x7fb7808f8ae0 to 0x7fb78c8b49e0: CUDA_ERROR_TOO_MANY_PEERS: peer mapping resources exhausted
2026-02-24 17:09:13.909745: W external/org_tensorflow/tensorflow/compiler/xla/pjrt/gpu/gpu_helpers.cc:63] Unable to enable peer access between GPUs 1 and 9; status: INTERNAL: failed to enable peer access from 0x7fb8008f89c0 to 0x7fb78c8b49e0: CUDA_ERROR_TOO_MANY_PEERS: peer mapping resources exhausted
2026-02-24 17:09:13.911764: W external/org_tensorflow/tensorflow/compiler/xla/pjrt/gpu/gpu_helpers.cc:63] Unable to enable peer access between GPUs 2 and 9; status: INTERNAL: failed to enable peer access from 0x7fb7908fb4c0 to 0x7fb78c8b49e0: CUDA_ERROR_TOO_MANY_PEERS: peer mapping resources exhausted
2026-02-24 17:09:13.913423: W external/org_tensorflow/tensorflow/compiler/xla/pjrt/gpu/gpu_

TypeError: 'NoneType' object is not iterable

In [14]:
# position_info = self.fetch_position_info(self.fetch1("KEY")).loc[
#                 time_slice
#             ]

key = (SortedSpikesDecodingV1 & {"nwb_file_name": "IM-1478_20220725_.nwb", "decoding_interval": "00_r1"}).fetch1("KEY")

display((SortedSpikesDecodingV1 & {"nwb_file_name": "IM-1478_20220725_.nwb", "decoding_interval": "00_r1"}).fetch_position_info(key))

[16:01:08][WARNING] Spyglass: Upsampled position data, frame indices are invalid. Setting add_frame_ind=False
/home/scrater/dev/spyglass/src/spyglass/decoding/v1/core.py:313: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  position_df[column][

(             position_x  position_y  orientation  velocity_x  velocity_y  \
 time                                                                       
 12.513374     90.645172  112.298968     2.592253    3.642322   -8.596940   
 12.515374     90.661739  112.287255     2.591226    3.605933   -8.414944   
 12.517374     90.678307  112.275542     2.590199    3.567587   -8.230246   
 12.519374     90.694874  112.263829     2.589172    3.527295   -8.042923   
 12.521374     90.711442  112.252116     2.588146    3.485071   -7.853053   
 ...                 ...         ...          ...         ...         ...   
 4876.159106   35.317122  131.472083     2.363328   -0.105210    0.686340   
 4876.161106   35.321200  131.476778     2.361982   -0.112388    0.690404   
 4876.163106   35.325277  131.481473     2.360634   -0.119628    0.694488   
 4876.165106   35.329354  131.486168     2.359286   -0.126922    0.698589   
 4876.167106   35.333432  131.490864     2.357937   -0.134262    0.702707   

In [ ]:
(SortedSpikesDecodingV1 & {"nwb_file_name": "IM-1478_20220725_.nwb", "decoding_interval": "00_r1"}).fetch_position_info(key).loc[]

[16:12:03][WARNING] Spyglass: Upsampled position data, frame indices are invalid. Setting add_frame_ind=False
/home/scrater/dev/spyglass/src/spyglass/decoding/v1/core.py:313: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  position_df[column][

pandas.core.frame.DataFrame

Populate `DecodedPosition`

This gets the most likely decoded x and y position at each time point.
Populating a single entry may take a long time, depending on how long your decoding interval is (30+ mins on breeze)

In [ ]:
from spyglass_hexmaze.hex_maze_decoding import DecodedPosition

# Create a key with the merge_id from DecodingOutput and the nwb_file_name
decoded_pos_key = {
    "decoding_merge_id": str(merge_id["merge_id"]),
    "nwb_file_name": nwb_file_name,
    "epoch": 0,
}
print(decoded_pos_key)

# Populate DecodedPosition
DecodedPosition.populate(decoded_pos_key)

{'decoding_merge_id': 'c6932302-5f74-dc1c-4e86-b1c3604817c3', 'nwb_file_name': 'IM-1478_20220725_.nwb', 'epoch': 0}


[2026-02-24 14:57:23,938][WARNING]: Skipped checksum for file with hash: ebc89d49-ba68-d908-d46e-cf0d9a342cee, and path: /stelmo/nwb/analysis/IM-1478_20220725/IM-1478_20220725_f62a2a49-1857-4819-b121-bec41242027c.nc
/home/scrater/miniforge3/envs/spyglass/lib/python3.10/site-packages/xarray/namedarray/core.py:496: UserWarning: Duplicate dimension names present: dimensions {'states'} appear more than once in dims=('states', 'states'). We do not yet support duplicate dimension names, but we do allow initial construction of the object. We recommend you rename the dims immediately to become distinct, as most xarray functionality is likely to fail silently if you do not. To rename the dimensions you will need to set the ``.dims`` attribute of each variable, ``e.g. var.dims=('x0', 'x1')``.
  warnings.warn(
/home/scrater/miniforge3/envs/spyglass/lib/python3.10/site-packages/xarray/namedarray/core.py:496: UserWarning: Duplicate dimension names present: dimensions {'states'} appear more than onc

Make sure it worked!

In [7]:
# Show our newly populated entry in the table
display(DecodedPosition() & decoded_pos_key)

# Fetch the df of max likelihood x,y decoded position
decoded_pos_df = (DecodedPosition & decoded_pos_key).fetch1_dataframe()
display(decoded_pos_df)

decoding_merge_id,nwb_file_name name of the NWB file,analysis_file_name name of the file,decoded_position_object_id
2ab779bc-2c30-5b6a-a343-d8f9e59aae17,IM-1478_20220720_.nwb,IM-1478_20220720_UYI4L03RTH.nwb,d83c217d-eab7-417c-9581-116b85678ac8


,hpd_thresh,spatial_cov,pred_x,pred_y
time,,,,
19.544026,0.014037,14,92.059067,49.903220
19.546026,0.010040,18,90.080151,47.916552
19.548026,0.010085,21,90.080151,47.916552
19.550026,0.008769,24,90.080151,47.916552
19.552026,0.007693,26,90.080151,47.916552
...,...,...,...,...
6789.257907,0.000373,1773,92.059067,71.756562
6789.259907,0.000373,1773,92.059067,71.756562
6789.261907,0.000373,1773,92.059067,71.756562


### Now assign this decoded position to a hex.

In [8]:
from spyglass_hexmaze.hex_maze_decoding import (
    DecodedHexPositionSelection,
    DecodedHexPosition,
)

# Insert into selection table!
DecodedHexPositionSelection.insert1(decoded_pos_key, skip_duplicates=True)

# Make sure it worked
display(DecodedHexPositionSelection() & decoded_pos_key)

decoding_merge_id,nwb_file_name name of the NWB file,epoch the session epoch for this task and apparatus(1 based)
2ab779bc-2c30-5b6a-a343-d8f9e59aae17,IM-1478_20220720_.nwb,0


In [11]:
# Run populate to assign to hex!
DecodedHexPosition.populate(decoded_pos_key)
DecodedHexPath.populate(decoded_pos_key)

# Make sure it worked
display(DecodedHexPosition() & decoded_pos_key)
display(DecodedHexPath() & decoded_pos_key)

[13:51:31][INFO] Spyglass: Writing new NWB file IM-1478_20220720_GIRZITV94S.nwb
INFO:spyglass:Writing new NWB file IM-1478_20220720_GIRZITV94S.nwb


decoding_merge_id,nwb_file_name name of the NWB file,epoch the session epoch for this task and apparatus(1 based),analysis_file_name name of the file,hex_assignment_object_id
2ab779bc-2c30-5b6a-a343-d8f9e59aae17,IM-1478_20220720_.nwb,0,IM-1478_20220720_1MLS6C5W8K.nwb,8be981cd-c5c4-45ab-8f87-720185ba5102


## Fetching data

In [10]:
# Fetch the df of decode assigned to hex
decoded_hex_df = (DecodedHexPosition & decoded_pos_key).fetch1_dataframe()
display(decoded_hex_df)

,hex,hex_including_sides,distance_from_centroid
time,,,
19.544026,8,8,2.649029
19.546026,8,8,3.553444
19.548026,8,8,3.553444
19.550026,8,8,3.553444
19.552026,8,8,3.553444
...,...,...,...
6789.257907,17,17,0.613474
6789.259907,17,17,0.613474
6789.261907,17,17,0.613474


In [11]:
# Fetch the combined hex and decoded position dataframe
combined_df = (DecodedHexPosition & decoded_pos_key).fetch_hex_and_position_dataframe()
display(combined_df)

,hpd_thresh,spatial_cov,pred_x,pred_y,hex,hex_including_sides,distance_from_centroid
time,,,,,,,
49.924743,0.000059,1226,123.824107,50.677172,9,9,4.717731
49.926743,0.000059,797,123.824107,50.677172,9,9,4.717731
49.928743,0.000045,225,125.811777,52.662603,9,9,2.455686
49.930743,0.000061,99,125.811777,52.662603,9,9,2.455686
49.932743,0.001249,27,125.811777,52.662603,9,9,2.455686
...,...,...,...,...,...,...,...
1663.932692,0.000166,1694,52.267987,138.036134,49,49_left,2.277534
1663.934692,0.000154,1622,52.267987,138.036134,49,49_left,2.277534
1663.936692,0.000158,1728,54.255657,138.036134,49,49_left,3.617987


## Below this is just other maybe helpful stuff but also you can ignore it.

---------------------------------------------
 Merge keys are hard sometimes. 
 
 I have a helper method `get_all_valid_keys` that finds all valid keys to insert into `DecodedHexPositionSelection`

Valid means the session exists in `HexMazeBlock`, `DecodedPosition`, and `HexCentroids`.

In [53]:
from spyglass_hexmaze.hex_maze_decoding import (
    DecodedHexPositionSelection,
    DecodedHexPosition,
)

# Get all valid keys for the DecodedHexPositionSelection table for this nwb
# (valid = the session has HexMazeBlock, DecodedPosition, and HexCentroids data)
all_valid_keys = DecodedHexPositionSelection.get_all_valid_keys(verbose=False)
nwb_file_keys = [key for key in all_valid_keys if key["nwb_file_name"] == nwb_file_name]

if not nwb_file_keys:
    print(f"No valid HexPositionSelection keys found for {nwb_file_name}")

# Insert each key into HexPositionSelection
for key in nwb_file_keys:

    # Skip inserting the key if it already exists in the table
    if key in DecodedHexPositionSelection:
        continue
    try:
        DecodedHexPositionSelection.insert1(key, skip_duplicates=True)
        print(f"Inserted new key {key} into DecodedHexPositionSelection")
    except Exception as e:
        print(f"Skipping insert for {key}: {e}")

In [54]:
# Only populate HexPosition with keys for this nwb
selection_keys = (DecodedHexPositionSelection & {"nwb_file_name": nwb_file_name}).fetch(
    "KEY"
)
print(selection_keys)
print(f"Populating HexPosition for {len(selection_keys)} entries in {nwb_file_name}")
DecodedHexPosition.populate(selection_keys)

[{'decoding_merge_id': UUID('fb231218-5693-1d21-6fcd-74d35ea7eefe'), 'nwb_file_name': 'IM-1478_20220726_.nwb', 'epoch': 0}]
Populating HexPosition for 1 entries in IM-1478_20220726_.nwb


{'success_count': 0, 'error_list': []}